# CHUYÊN ĐỀ TOÀN DIỆN: CÁC KỸ THUẬT TỐI ƯU HÓA SPARK & DELTA LAKE DÀNH CHO DATA ENGINEER (PROD-READY)

Notebook này được thiết kế theo tiêu chuẩn thực nghiệm khắt khe trong môi trường Enterprise:
- **Loại bỏ sai số Cold-Start (Làm nóng JVM / Spark Warm-up trước khi đo):** Đảm bảo kết quả benchmark phản ánh đúng 100% hiệu năng thuật toán.
- **So sánh Đối Đầu (Before vs After):** Mỗi kỹ thuật đều có 2 kịch bản chạy song song để bạn thấy rõ sự khác biệt về **Physical Plan**, **Thời gian thực thi**, và **Phân bố dữ liệu trên Partition**.
- **Đầy đủ 6 Kỹ Thuật Trọng Điểm:**
  1. **Broadcast Join vs Sort-Merge Join:** Triệt tiêu Network Shuffle.
  2. **Adaptive Query Execution (AQE):** Tự động gom Partition lúc runtime.
  3. **Xử lý Data Skew bằng kỹ thuật Salting:** Minh họa trực quan phân bố dòng trên từng partition.
  4. **`coalesce()` vs `repartition()`:** Tối ưu hóa ghi file và loại bỏ Shuffle.
  5. **Delta Lake Compaction & Z-ORDER:** Giải quyết Small Files và Data Skipping.
  6. **Delta Lake `VACUUM`:** Dọn dẹp file mồ côi kiểm soát chi phí storage.

## BƯỚC 0: KHỞI ĐỘNG VÀ LÀM NÓNG MÁY ẢO JVM (SPARK WARM-UP)

> **Tại sao cần Warm-up?** 
> Khi Spark vừa khởi động, các Thread Pool của Executor chưa được cấp phát, JIT Compiler của Java chưa nạp Bytecode. Lệnh đầu tiên luôn bị trễ thêm 1-2 giây vì "Cold Start". Ta chạy một lệnh giả lập để JVM sẵn sàng trước khi thực hiện các bài benchmark.

In [ ]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("--- ĐANG THỰC HIỆN SPARK & JVM WARM-UP ---")
t_start = time.time()
_ = spark.range(0, 50000).groupBy().count().collect()
print(f"-> JVM và Spark Context đã sẵn sàng (Thời gian khởi động: {time.time() - t_start:.2f}s)!
")

# Hàm tiện ích đo thời gian chạy công bằng (chạy 1 lần làm ấm, lấy trung bình 2 lần sau)
def benchmark_action(action_fn, name="Action", runs=2):
    # Lần 1: Chạy để đưa dữ liệu vào pipeline cache nếu có
    action_fn()
    # Các lần sau: Đo đạc chính thức
    times = []
    for _ in range(runs):
        t0 = time.time()
        action_fn()
        times.append(time.time() - t0)
    avg_time = sum(times) / len(times)
    print(f"[{name}] Thời gian trung bình: {avg_time:.4f} giây")
    return avg_time


## KỸ THUẬT 1: BROADCAST HASH JOIN VS SORT-MERGE JOIN

### Thử nghiệm chuẩn xác:
Mặc định Spark tự động Broadcast các bảng < 10MB. Để thấy rõ sự khác biệt của **Sort-Merge Join (Shuffle cả 2 bảng)** và **Broadcast Join (Không Shuffle)**, ta sẽ:
1. Tắt tự động Broadcast (`autoBroadcastJoinThreshold = -1`) $ightarrow$ Ép Spark dùng **Sort-Merge Join**.
2. Bật lại Hint `broadcast()` $ightarrow$ Ép Spark dùng **Broadcast Hash Join**.

In [ ]:
# Tạo 1 bảng Fact 200,000 dòng và 1 bảng Dim 1,000 dòng
df_fact = spark.range(0, 200000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), (F.col("id") % 1000).cast("string"))) \
    .withColumn("revenue", F.rand() * 1000)

df_dim = spark.range(0, 1000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), F.col("id").cast("string"))) \
    .withColumn("customer_group", F.when(F.col("id") % 2 == 0, "RETAIL").otherwise("WHOLESALE"))

# ==============================================================================
# 1.1 ÉP BUỘC SORT-MERGE JOIN (BẰNG CÁCH TẮT AUTO BROADCAST)
# ==============================================================================
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1) # Tắt broadcast tự động
df_smj = df_fact.join(df_dim, "customer_id")

print("=== 1.1 KẾ HOẠCH SORT-MERGE JOIN (CÓ SHUFFLE CẢ 2 BẢNG) ===")
df_smj.explain(False)
t_smj = benchmark_action(lambda: df_smj.count(), name="Sort-Merge Join")

# ==============================================================================
# 1.2 DÙNG HINT BROADCAST() (TRIỆT TIÊU TOÀN BỘ SHUFFLE)
# ==============================================================================
df_bhj = df_fact.join(F.broadcast(df_dim), "customer_id")

print("\n=== 1.2 KẾ HOẠCH BROADCAST HASH JOIN (KHÔNG SHUFFLE) ===")
df_bhj.explain(False)
t_bhj = benchmark_action(lambda: df_bhj.count(), name="Broadcast Hash Join")

# Khôi phục lại ngưỡng mặc định 10MB của Spark
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

print(f"\n-> KẾT QUẢ: Broadcast Join nhanh hơn gấp {t_smj / t_bhj:.2f} lần so với Sort-Merge Join!")


## KỸ THUẬT 2: ADAPTIVE QUERY EXECUTION (AQE) - TỰ ĐỘNG GỘP SHUFFLE PARTITIONS

### Vấn đề Partition Mặc Định:
Mặc định Spark đặt `spark.sql.shuffle.partitions = 200`. Với các truy vấn tổng hợp vừa và nhỏ, 200 partition khiến Spark chia nhỏ dữ liệu thành 200 task tí hon, làm nghẽn Scheduler.

### Cơ chế AQE:
Khi bật AQE, Spark tự quan sát kích thước thực tế sau khi Shuffle và tự động gộp (Coalesce) 200 partition xuống còn đúng số lượng cần thiết (ví dụ 4-8 partition)!

In [ ]:
# 2.1 TRƯỜNG HỢP KHÔNG CÓ AQE (TẮT AQE)
spark.conf.set("spark.sql.adaptive.enabled", "false")
df_no_aqe = df_fact.groupBy("customer_id").agg(F.sum("revenue").alias("total"))
df_no_aqe.collect() # Trigger action
partitions_no_aqe = df_no_aqe.rdd.getNumPartitions()
print(f"[KHÔNG BẬT AQE] Số lượng Shuffle Partitions bị cố định: {partitions_no_aqe} partitions (Gây lãng phí tài nguyên)")

# 2.2 TRƯỜNG HỢP BẬT AQE TỐI ƯU
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "1MB")

df_with_aqe = df_fact.groupBy("customer_id").agg(F.sum("revenue").alias("total"))
df_with_aqe.collect()
partitions_aqe = df_with_aqe.rdd.getNumPartitions()
print(f"[ĐÃ BẬT AQE]   Số lượng Shuffle Partitions tự động gộp tối ưu: {partitions_aqe} partitions!")

print("\n--- Kế hoạch AdaptiveQuery (Notice: CoalescedShuffleSpec): ---")
df_with_aqe.explain(False)


## KỸ THUẬT 3: XỬ LÝ LỆCH DỮ LIỆU (DATA SKEW) BẰNG KỸ THUẬT SALTING

### Minh họa hiện tượng nghẽn cổ chai:
- Giả lập: 90% đơn hàng thuộc về 1 khách hàng VIP duy nhất (`CUST_SKEWED`).
- Khi Join thường: 1 Executor phải xử lý 90,000 dòng trong khi các Executor khác chỉ nhận vài trăm dòng.
- Dùng hàm `spark_partition_id()` để soi trực tiếp số dòng trên từng Partition trước và sau khi Salting!

In [ ]:
# 1. Tạo tập dữ liệu bị Skew nặng (90% dòng là CUST_SKEWED)
df_skewed_data = spark.range(0, 100000) \
    .withColumn("customer_id", F.when(F.col("id") < 90000, "CUST_SKEWED").otherwise(F.concat(F.lit("CUST_"), F.col("id")))) \
    .withColumn("amount", F.rand() * 500)

df_lookup_dim = spark.createDataFrame([
    ("CUST_SKEWED", "VIP Super Client"),
    ("CUST_95000", "Regular Client")
], ["customer_id", "tier"])

# 3.1 KHI CHƯA SALTING: SOI SỰ LỆCH DỮ LIỆU TRÊN CÁC PARTITION
df_unbalanced = df_skewed_data.repartition(4, "customer_id")
print("=== PHÂN BỐ DÒNG KHI CHƯA SALTING (1 PARTITION BỊ QUÁ TẢI NẶNG) ===")
df_unbalanced.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

# 3.2 ÁP DỤNG KỸ THUẬT SALTING (CHIA NHỎ THÀNH 4 NHÁNH MUỐI)
SALT_FACTOR = 4

# Bảng lớn: ghép salt ngẫu nhiên 0..3
df_skewed_salted = df_skewed_data \
    .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt")))

# Bảng nhỏ: nhân bản explode đủ 4 salt
df_dim_salted = df_lookup_dim \
    .withColumn("salt_array", F.array([F.lit(i) for i in range(SALT_FACTOR)])) \
    .withColumn("salt", F.explode("salt_array")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt"))) \
    .drop("salt_array")

# Phân bổ lại theo salted_key
df_balanced = df_skewed_salted.repartition(4, "salted_key")
print("=== PHÂN BỐ DÒNG SAU KHI SALTING (DỮ LIỆU ĐƯỢC CHIA ĐỀU BẰNG NHAU CHO 4 PARTITION) ===")
df_balanced.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()


## KỸ THUẬT 4: `coalesce()` VS `repartition()` KHI GHI FILE

### Nguyên tắc cốt tử:
- **`coalesce(n)`:** Chỉ gom các partition kế cận trên cùng node $ightarrow$ **0% Shuffle**, cực nhanh khi muốn giảm số file trước khi ghi đĩa.
- **`repartition(n)`:** Xáo trộn toàn bộ dữ liệu qua mạng (**Full Shuffle**) $ightarrow$ Tốn CPU/Network, chỉ dùng khi muốn tăng độ song song.

In [ ]:
df_wide = spark.range(0, 300000).repartition(16)
print(f"Số partition ban đầu: {df_wide.rdd.getNumPartitions()}
")

# Đo coalesce(4) - KHÔNG SHUFFLE
t_coalesce = benchmark_action(lambda: df_wide.coalesce(4).count(), name="coalesce(4)")

# Đo repartition(4) - CÓ SHUFFLE TOÀN BỘ
t_repartition = benchmark_action(lambda: df_wide.repartition(4).count(), name="repartition(4)")

print(f"\n-> KẾT QUẢ: coalesce() chạy nhanh hơn repartition() gấp {t_repartition / t_coalesce:.2f} lần mà không tốn chi phí Shuffle mạng!")


## KỸ THUẬT 5: DELTA LAKE COMPACTION & Z-ORDER CLUSTERING

### Thử nghiệm giải quyết Small Files Problem:
1. Giả lập ghi nhiều mẻ nhỏ $ightarrow$ Sinh ra nhiều file Parquet tí hon.
2. Chạy `OPTIMIZE ... ZORDER BY (customer_id)` $ightarrow$ Gom file và sắp xếp theo đường cong lấp đầy không gian Z-Order để tối ưu Data Skipping.

In [ ]:
tbl_opt_demo = "opt_compaction_zorder_demo"

# Giả lập ghi 8 partition nhỏ ra Delta Lake
spark.range(0, 50000) \
    .withColumn("customer_id", F.concat(F.lit("CUST_"), (F.col("id") % 500).cast("string"))) \
    .withColumn("order_val", F.rand() * 1000) \
    .repartition(8) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl_opt_demo)

print("=== TRƯỚC KHI TỐI ƯU HÓA ===")
spark.sql(f"DESCRIBE DETAIL {tbl_opt_demo}").select("numFiles", "sizeInBytes").show()

# Thực hiện Gom file (Compaction) và Sắp xếp Z-Order
print("--- Đang thực thi OPTIMIZE kèm Z-ORDER BY (customer_id)... ---")
spark.sql(f"OPTIMIZE {tbl_opt_demo} ZORDER BY (customer_id)").show(truncate=False)

print("=== SAU KHI TỐI ƯU HÓA (CÁC FILE NHỎ ĐÃ GỘP THÀNH FILE CHUẨN) ===")
spark.sql(f"DESCRIBE DETAIL {tbl_opt_demo}").select("numFiles", "sizeInBytes").show()


## KỸ THUẬT 6: QUẢN LÝ DỌN DẸP BẢN GHI RÁC BẰNG DELTA LAKE `VACUUM`

Khi dữ liệu bị sửa (`UPDATE`) hoặc gộp (`OPTIMIZE`), các file Parquet cũ trở thành "file rác" (tombstone). Sau 7-30 ngày, lệnh **`VACUUM`** sẽ xóa vĩnh viễn các file này để giải phóng dung lượng đĩa.

In [ ]:
# Tắt ngưỡng an toàn 7 ngày (Chỉ dùng cho mục đích Demo/Lab để thấy file bị xóa ngay lập tức):
spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print(f"--- ĐANG THỰC HIỆN VACUUM TRÊN BẢNG {tbl_opt_demo} ---")
spark.sql(f"VACUUM {tbl_opt_demo} RETAIN 0 HOURS").show(truncate=False)

print("Đã dọn sạch toàn bộ các file Parquet mồ côi cũ! Bảng đạt trạng thái lưu trữ gọn gàng nhất.")
